# TAPA tutorial — speaker diarization + phonetic analysis

TAPA takes a recording (a file or a YouTube URL) and produces **per-speaker
phonetic measurements**: vowel formants, stop voice-onset time (VOT), and
fricative spectral moments.

The pipeline runs six stages:

| stage | what it does | tool |
|---|---|---|
| 1. Diarization | find speech, cluster it by speaker | Silero VAD + Resemblyzer |
| 2. Transcription | word-level text with timestamps | Whisper |
| 3. Forced alignment | place phoneme boundaries | MFA (or a CMUdict fallback) |
| 4. Segment identification | label vowels / stops / fricatives | ARPABET maps |
| 5. Acoustic measurement | formants, VOT, spectral moments | Praat (or Dr.VOT for stops) |
| 6. Aggregation | per-speaker averages + outlier rejection | — |

**Before you start:** choose *Runtime → Change runtime type → GPU*. The pipeline
runs on CPU too, but Whisper is several times slower.

Everything below is configured in **one settings cell**, so you can run the
notebook end-to-end untouched and then change individual options.

## 1. Settings

Pick a **preset**, then override individual options in the cell below it if you
want. The presets differ only in accuracy-vs-time trade-offs.

| preset | alignment | stop VOT | Whisper | ~time for 2 min of audio | best for |
|---|---|---|---|---|---|
| `quick` | CMUdict (approximate) | Praat | `small.en` | ~2 min | a first look; checking your audio runs end-to-end |
| `mfa_praat` | MFA | Praat | `small.en` | ~7 min | vowel formants and fricatives — precise boundaries, no Dr.VOT setup |
| `recommended` | MFA | Dr.VOT | `small.en` | ~10 min | most studies, and anything involving VOT |
| `max_accuracy` | MFA | Dr.VOT | `medium.en` | ~13 min | difficult audio: accents, noise, overlapping speech |
| `drvot_no_mfa` | CMUdict (approximate) | Dr.VOT | `small.en` | ~5 min | VOT work when MFA will not install |

Times include the one-time MFA (~5 min) and Dr.VOT (~2 min) setup where needed;
later runs in the same session skip it.

The cell below starts on **`quick`** so your first run finishes in a couple of
minutes and you can see the whole pipeline and its outputs. Switch to
`recommended` (or `mfa_praat` if you don't need VOT) for real analysis — the
CMUdict alignment behind `quick` is too approximate to publish from.

**Choosing between them.** The two decisions are independent. *Alignment* sets
how precisely phoneme boundaries are placed — MFA is a real forced aligner,
while the CMUdict fallback just distributes phonemes proportionally across each
word, which is fine for a smoke test but too coarse for publication. *Stop VOT*
only affects the stop measurements: vowel and fricative results are identical
either way, so if VOT is not part of your research question, `mfa_praat` saves
the Dr.VOT setup entirely.

A note on stop VOT: the Praat backend is fast and needs no setup, but it is
sensitive to burst placement in conversational speech. **For any study whose
conclusions depend on VOT, use the Dr.VOT backend** (`vot_backend="drvot"`),
which is a model trained on hand-labeled VOTs and also handles prevoicing
(negative VOT). Vowel formants and fricative measurements are unaffected by
this choice.

In [ ]:
# "quick" | "mfa_praat" | "recommended" | "max_accuracy" | "drvot_no_mfa"
PRESET = "quick"

PRESETS = {
    "quick":         dict(use_mfa=False, vot_backend="tapa",  whisper_model="small.en"),
    "mfa_praat":     dict(use_mfa=True,  vot_backend="tapa",  whisper_model="small.en"),
    "recommended":   dict(use_mfa=True,  vot_backend="drvot", whisper_model="small.en"),
    "max_accuracy":  dict(use_mfa=True,  vot_backend="drvot", whisper_model="medium.en"),
    "drvot_no_mfa":  dict(use_mfa=False, vot_backend="drvot", whisper_model="small.en"),
}
S = dict(PRESETS[PRESET])

# ---- Audio source -----------------------------------------------------------
# "sample"      : a 2-minute clip hosted with the repo (good for a first run)
# "sample_1.5h" : a 90-minute recording, for long-input testing
# "sample_2h"   : a 2-hour recording, the longest we publish
# "upload"      : upload your own file when prompted
# "youtube"     : paste a URL below
AUDIO_SOURCE = "sample"
YOUTUBE_URL  = "https://www.youtube.com/watch?v=DPO7imV0LHg"

# ---- Overrides (uncomment to change) ---------------------------------------
# S["whisper_model"] = "medium.en"   # tiny.en | base.en | small.en | medium.en | large
# S["vot_backend"]   = "tapa"        # "tapa" (Praat) or "drvot" (neural, recommended)
# S["use_mfa"]       = False         # False -> CMUdict proportional timing

# Diarization
NUM_SPEAKERS      = None   # None = auto-detect; SET THIS if you know the count
MIN_SPEAKER_SHARE = 0.0    # 0 = off. e.g. 0.02 absorbs any speaker holding
                           # under 2% of the speech into the nearest one —
                           # useful when auto-detection splits one talker
MERGE_GAP         = 0.5    # seconds; merge same-speaker segments closer than this

# Measurement thresholds (defaults match the published pipeline)
MIN_VOWEL_DURATION = 0.03     # s; shorter vowels are skipped
TARGET_VOWELS      = None     # e.g. {"i", "u", "ɑ"} to restrict to specific IPA vowels
MAD_THRESHOLD      = 2.0      # outlier rejection strength (higher = more permissive)

# Alignment memory/speed (matter on long recordings)
MFA_SPLIT_UTTERANCES = True   # align per diarization segment; keeps MFA memory bounded
MFA_NUM_JOBS         = 2      # MFA workers; 2 suits Colab's 2 vCPUs

print(f"preset={PRESET}: {S}")

## 2. Install

`ffmpeg` and TAPA are always installed. MFA and Dr.VOT are installed only if
your settings need them — each adds several minutes the first time.

In [ ]:
# TAPA_BRANCH must match the branch this notebook came from, otherwise you
# install a different version of the pipeline than the notebook expects.
TAPA_BRANCH = "colab-memory-fixes"

!apt-get install -y -qq ffmpeg > /dev/null
!pip install -q "git+https://github.com/sarmadchandio/tapa.git@{TAPA_BRANCH}"

import tapa, inspect, os
print("TAPA installed from branch:", TAPA_BRANCH)
print("   package:", os.path.dirname(inspect.getfile(tapa)))
try:
    from tapa.audio import load_audio_16k          # present only with the memory fixes
    print("   single-decode audio path: present")
except ImportError:
    print("   WARNING: this build decodes the audio several times and will run out "
          "of RAM on recordings longer than about an hour.")

In [ ]:
# Montreal Forced Aligner (only when use_mfa=True)
import os
if S["use_mfa"]:
    if not os.path.exists("/opt/miniforge/bin/mfa"):
        !wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/mf.sh
        !bash /tmp/mf.sh -b -p /opt/miniforge > /dev/null
        !/opt/miniforge/bin/mamba install -y -q -c conda-forge montreal-forced-aligner > /dev/null
    !/opt/miniforge/bin/mfa model download acoustic english_us_arpa 2>/dev/null
    !/opt/miniforge/bin/mfa model download dictionary english_us_arpa 2>/dev/null
    print("MFA ready")
else:
    print("skipped (use_mfa=False)")

In [ ]:
# Dr.VOT (only when vot_backend="drvot")
# Dr.VOT calls praat to extract pitch and intensity, then sox while building
# its acoustic features — both must be present or every token silently falls
# back to the Praat estimator. We install Praat's "barren" build, which is
# headless: the GUI build Dr.VOT bundles needs GTK 2 and fails on Colab.
PRAAT_BARREN = ("https://github.com/praat/praat/releases/download/v6.4.30/"
                "praat6430_linux-intel64-barren.tar.gz")
if S["vot_backend"] == "drvot":
    !pip install -q "tapa[drvot] @ git+https://github.com/sarmadchandio/tapa.git@{TAPA_BRANCH}"
    !apt-get install -y -qq sox > /dev/null
    !wget -q -O /tmp/praat.tar.gz {PRAAT_BARREN}
    !tar xzf /tmp/praat.tar.gz -C /tmp && mv -f /tmp/praat_barren /usr/local/bin/praat && chmod +x /usr/local/bin/praat
    !praat --version
    !python -m tapa.drvot setup /content/Dr.VOT
    print("Dr.VOT ready")
else:
    print("skipped (vot_backend='tapa')")

## 3. Preflight check

Confirms the environment before you spend time on a long run. The dependency
check is the same one you can run yourself at any time with
`python -m tapa.environment`; anything it reports as missing means the
corresponding stage will fail or silently fall back to a cruder method.

In [ ]:
from tapa.environment import check_environment, report

checks = check_environment(vot_backend=S["vot_backend"],
                           drvot_repo="/content/Dr.VOT" if S["vot_backend"] == "drvot" else None,
                           mfa_bin="/opt/miniforge/bin/mfa" if S["use_mfa"] else None)
print("Dependencies:")
report(checks)

In [ ]:
import shutil, torch, psutil, os

gpu = torch.cuda.get_device_name() if torch.cuda.is_available() else None
# Binary units (GiB), so these match the figure Colab shows in its resource
# meter — Colab labels them "GB" but the number is GiB.
GB = 2 ** 30
ram = psutil.virtual_memory().total / GB
mfa_ok = os.path.exists("/opt/miniforge/bin/mfa")
drvot_ok = os.path.exists("/content/Dr.VOT/final_models/adv_model.model")

print(f"{'GPU':<12}: {gpu or 'none (CPU only - Whisper will be slow)'}")
print(f"{'RAM':<12}: {ram:.1f} GB")
print(f"{'ffmpeg':<12}: {'found' if shutil.which('ffmpeg') else 'MISSING'}")
print(f"{'MFA':<12}: {'found' if mfa_ok else 'not installed'}"
      + ("  <-- needed by your settings" if S["use_mfa"] and not mfa_ok else ""))
print(f"{'Dr.VOT':<12}: {'found' if drvot_ok else 'not installed'}"
      + ("  <-- needed by your settings" if S["vot_backend"] == "drvot" and not drvot_ok else ""))

print("\nRough timing guide (GPU runtime):")
print("  Whisper + diarization : ~1x audio duration with small.en")
print("  MFA alignment         : ~0.5x audio duration")
print("  Dr.VOT                : ~1 s per stop token (~450 tokens per 10 min of speech)")

print("\nMemory (measured, GPU node, no MFA):")
print("   20 min -> 1.9 GB      60 min -> 2.5 GB      120 min -> 4.3 GB")
print("The audio is decoded once at 16 kHz mono and shared by every stage, so")
print("the baseline stays near 1.6 GB; the peak comes from Whisper building its")
print("spectrogram for the whole recording at the start of transcription, and")
print("that is what grows with duration (~1.8 GB per hour). A 2-hour recording")
print("therefore fits comfortably; the ceiling on a standard Colab runtime is")
print("roughly 5-6 hours. Try AUDIO_SOURCE='sample_1.5h' or 'sample_2h'.")

## 4. Choose the audio

In [ ]:
import os
SAMPLES = {"sample": "tutorial_2min.mp3", "sample_1.5h": "sample_1.5h.mp3",
           "sample_2h": "sample_2h.mp3"}
if AUDIO_SOURCE in SAMPLES:
    AUDIO = SAMPLES[AUDIO_SOURCE]
    if not os.path.exists(AUDIO):
        !wget -q https://github.com/sarmadchandio/tapa/releases/download/bench-audio/{AUDIO}
elif AUDIO_SOURCE == "upload":
    from google.colab import files
    AUDIO = list(files.upload().keys())[0]
elif AUDIO_SOURCE == "youtube":
    AUDIO = YOUTUBE_URL      # the pipeline downloads URLs itself
else:
    raise ValueError(AUDIO_SOURCE)
print("audio:", AUDIO)

If a YouTube download fails with *"Sign in to confirm you're not a bot"*,
export a `cookies.txt` from a logged-in browser (the *Get cookies.txt LOCALLY*
extension), upload it via the folder icon in the sidebar, and re-run — TAPA
finds `/content/cookies.txt` automatically.

## 5. Run the pipeline

`TAPAConfig` holds every option; `pipeline.run()` executes the six stages and
writes the result files to `results/`.

In [ ]:
from tapa.config import TAPAConfig
from tapa.pipeline import TAPAPipeline

cfg_kwargs = dict(
    results_dir="results/",
    whisper_model=S["whisper_model"],
    vot_backend=S["vot_backend"],
    num_speakers=NUM_SPEAKERS,
    min_speaker_share=MIN_SPEAKER_SHARE,
    merge_gap=MERGE_GAP,
    min_vowel_duration=MIN_VOWEL_DURATION,
    target_vowels=TARGET_VOWELS,
    mad_threshold=MAD_THRESHOLD,
)
if S["use_mfa"]:
    cfg_kwargs.update(mfa_bin="/opt/miniforge/bin/mfa",
                      mfa_split_utterances=MFA_SPLIT_UTTERANCES,
                      mfa_num_jobs=MFA_NUM_JOBS)
if S["vot_backend"] == "drvot":
    cfg_kwargs["drvot_repo_dir"] = "/content/Dr.VOT"

cfg = TAPAConfig(**{k: v for k, v in cfg_kwargs.items()
                    if k in TAPAConfig.__dataclass_fields__})
pipeline = TAPAPipeline(config=cfg)

# Print memory use every 30 s. On long recordings this shows how much headroom
# you have, and if the runtime does run out, the last line says which stage.
import threading, time, psutil
_GB = 2 ** 30          # GiB — matches the number Colab's resource meter shows
_stop = threading.Event()
def _watch():
    proc = psutil.Process()
    while not _stop.wait(30):
        used = proc.memory_info().rss
        for c in proc.children(recursive=True):
            try: used += c.memory_info().rss
            except psutil.Error: pass
        total = psutil.virtual_memory().total
        print(f"   [mem] {used/_GB:.1f} GB of {total/_GB:.1f} GB in use", flush=True)
threading.Thread(target=_watch, daemon=True).start()

try:
    results = pipeline.run(AUDIO)
finally:
    _stop.set()

### Reading the run log

Three things in the output above are worth checking before you trust the
numbers.

**Which alignment ran.** Step 4 prints `alignment method:` and the `[DONE]`
line repeats it. If you asked for MFA but see *CMUdict proportional timing
(approximate — MFA did not run)*, alignment failed and the run fell back to
approximate boundaries; the underlying error appears a few lines earlier.

**How the speakers came out.** Step 1 lists each speaker with their share of
the speech. Automatic speaker-count estimation sometimes splits one talker into
two, and the giveaway is a speaker holding only a percent or two — the log flags
this when it happens. If you know how many people are in the recording, set
`NUM_SPEAKERS` above; that constrains clustering and is more reliable than any
heuristic. `MIN_SPEAKER_SHARE` is the automatic alternative.

**Which words were dropped.** Step 4 reports how many words received no
phoneme-level alignment. MFA marks words missing from its pronunciation
dictionary as `spn` — typically numerals ("2016"), initialisms ("phd"), and
proper nouns — and those words contribute no phonemes, so they are absent from
every measurement. A couple of percent is normal; a high figure means the
dictionary is a poor fit for your material and you should supply a
supplementary pronunciation dictionary to MFA.

## 6. Explore the results

Every stage writes a file you can open in pandas, R, or Excel:

| file | contents |
|---|---|
| `*_diarization.csv` | speaker turns (speaker, start, end) |
| `*_transcription.csv` / `.txt` | words with timestamps and speaker |
| `*_vowel_averages.csv` | per-speaker, per-vowel mean F1/F2 |
| `*_vowel_formants.json` | every individual vowel token |
| `*_stop_averages.csv` | per-speaker, per-stop mean VOT |
| `*_stop_vot.json` | every stop token |
| `*_fricative_averages.csv` | per-speaker spectral moments |
| `*_fricative_spectra.json` | every fricative token |

In [ ]:
import glob, pandas as pd
for f in sorted(glob.glob("results/*")):
    print(f)

stem = os.path.splitext(os.path.basename(sorted(glob.glob("results/*_diarization.csv"))[0]))[0]
stem = stem.replace("_diarization", "")
vowels = pd.read_csv(f"results/{stem}_vowel_averages.csv")
vowels.head(10)

In [ ]:
# Vowel space: F1 vs F2 per speaker (phonetics convention: both axes reversed)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5.5))
for spk, grp in vowels.groupby("speaker"):
    ax.scatter(grp["mean_f2"], grp["mean_f1"], s=grp["n_tokens"] * 3, alpha=.6, label=spk)
    for _, r in grp.iterrows():
        ax.annotate(r["vowel"], (r["mean_f2"], r["mean_f1"]),
                    ha="center", va="center", fontsize=9)
ax.invert_xaxis(); ax.invert_yaxis()
ax.set_xlabel("F2 (Hz)"); ax.set_ylabel("F1 (Hz)")
ax.set_title("Vowel space by speaker (marker size = token count)")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Stop VOT and fricative spectra
stops = pd.read_csv(f"results/{stem}_stop_averages.csv")
frics = pd.read_csv(f"results/{stem}_fricative_averages.csv")
print("Stop VOT (ms):"); display(stops)
print("Fricative spectral moments:"); display(frics.head(10))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for spk, grp in stops.groupby("speaker"):
    axes[0].bar(grp["phone"] + f"\n{spk[-2:]}", grp["mean_vot_ms"], alpha=.75, label=spk)
axes[0].set_ylabel("mean VOT (ms)"); axes[0].set_title("Stop VOT"); axes[0].legend(fontsize=8)
for spk, grp in frics.groupby("speaker"):
    axes[1].bar(grp["phone"] + f"\n{spk[-2:]}", grp["mean_cog"], alpha=.75, label=spk)
axes[1].set_ylabel("centre of gravity (Hz)"); axes[1].set_title("Fricative spectra")
plt.tight_layout(); plt.show()

**Reading the VOT table:** English voiceless stops (/p t k/) are normally
longer than voiced (/b d g/), and in careful speech aspirated voiceless stops
run tens of milliseconds. Values clustered near zero, or no voiced/voiceless
separation, mean burst detection struggled with your recording — switch to
`vot_backend="drvot"` and re-run.

With Dr.VOT each token in `*_stop_vot.json` also carries `vot_method`
(`"drvot"` or `"tapa-fallback"`) and a `vot_type` of `POS` (aspirated) or
`NEG` (prevoiced), so you can filter on how each measurement was obtained.

## 7. Save your results

In [ ]:
!zip -qr tapa_results.zip results
from google.colab import files
files.download("tapa_results.zip")

### What is inside the download

Every file you just downloaded, previewed below, so you know the exact shape of
the data before opening it in R, Excel, or your own scripts. The CSVs hold the
**aggregated** results; the JSONs hold **every individual token** behind those
averages, which is what you want for statistics.

In [ ]:
import glob, json, os
import pandas as pd
from IPython.display import display

stem = os.path.basename(sorted(glob.glob("results/*_diarization.csv"))[0]).replace("_diarization.csv", "")

print("results/ contents:")
for f in sorted(glob.glob("results/*")):
    print(f"   {os.path.basename(f):<45} {os.path.getsize(f)/1024:8.1f} KB")

In [ ]:
# The aggregated tables (CSV)
for name, note in [
    ("diarization",         "who spoke when"),
    ("transcription",       "each word, with timing and speaker"),
    ("vowel_averages",      "mean F1/F2 per speaker per vowel"),
    ("stop_averages",       "mean VOT per speaker per stop"),
    ("fricative_averages",  "spectral moments per speaker per fricative"),
]:
    path = f"results/{stem}_{name}.csv"
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    print(f"\n=== {os.path.basename(path)} — {note}  ({len(df)} rows) ===")
    display(df.head(8))

In [ ]:
# The readable transcript
txt = f"results/{stem}_transcription.txt"
if os.path.exists(txt):
    print(f"=== {os.path.basename(txt)} (first 20 lines) ===\n")
    print("".join(open(txt).readlines()[:20]))

In [ ]:
# The token-level records (JSON) — one entry per measured segment
for name, note in [
    ("vowel_formants",    "one record per vowel token"),
    ("stop_vot",          "one record per stop token"),
    ("fricative_spectra", "one record per fricative token"),
]:
    path = f"results/{stem}_{name}.json"
    if not os.path.exists(path):
        continue
    data = json.load(open(path))
    total = sum(len(toks) for per_phone in data.values() for toks in per_phone.values())
    spk = next(iter(data)); phone = next(iter(data[spk]))
    print(f"\n=== {os.path.basename(path)} — {note} ===")
    print(f"structure: {{speaker: {{phoneme: [tokens]}}}}   "
          f"speakers={list(data)}   total tokens={total}")
    print(f"example — {spk} / '{phone}', first of {len(data[spk][phone])} tokens:")
    print(json.dumps(data[spk][phone][0], indent=2))

In [ ]:
# Turn the token-level JSON into a flat dataframe — the usual starting point
# for statistics (one row per measured token).
rows = []
for spk, per_vowel in json.load(open(f"results/{stem}_vowel_formants.json")).items():
    for vowel, tokens in per_vowel.items():
        for t in tokens:
            rows.append({"speaker": spk, "vowel": vowel, **t})
tokens_df = pd.DataFrame(rows)
print(f"{len(tokens_df)} vowel tokens, {tokens_df.shape[1]} columns")
display(tokens_df.head(10))
# e.g. tokens_df.to_csv("vowel_tokens_flat.csv", index=False)

## Where to go next

- **Longer recordings:** the pipeline streams audio at 16 kHz mono and aligns
  per segment, so memory stays flat; expect runtime to scale with duration.
- **Batch processing:** `pipeline.run_batch("audio_dir/")` processes a folder.
- **Command line:** `tapa recording.mp3 -o results/ --vot-backend drvot --drvot-repo /content/Dr.VOT`
- **All options:** see `TAPAConfig` in
  [`tapa/config.py`](https://github.com/sarmadchandio/tapa/blob/master/tapa/config.py)
  and the [README](https://github.com/sarmadchandio/tapa#readme).

If you use TAPA in published work, please cite the repository and — when using
the Dr.VOT backend — Shrem, Goldrick & Keshet (2019), *Dr.VOT: Measuring
Positive and Negative Voice Onset Time in the Wild*.